# A Bayesian Davidson–Bradley–Terry Model for Titled Tuesday

The MAP version of this model was the best performer in `modeling.ipynb` (test log loss **0.7564** vs Elo-only **0.7801**). This notebook is the full Bayesian treatment:

1. **Model specification** in PyMC (non-centered, Glicko-informed priors) + prior predictive checks
2. **NUTS inference** and convergence diagnostics
3. **Posterior analysis** — white advantage, draw propensity, skill shrinkage
4. **Posterior predictive evaluation** — does marginalizing over the posterior beat the MAP plug-in?
5. **Model improvements as ablations** — rating-dependent draws, form covariates, hierarchical title layer
6. **Model comparison** — test log loss across variants
7. **Interpretability** — who over/underperforms their rating; per-game prediction decomposition; epistemic vs aleatoric uncertainty
8. **Production design** (prose)

**The model.** Player $i$ has latent skill $\theta_i$, strength $\pi_i = e^{\theta_i + \gamma\,\mathbb{1}[i=\text{white}]}$, and outcomes follow Davidson (1970):

$$P(\text{white win}) = \frac{\pi_w}{D}, \quad P(\text{draw}) = \frac{\nu\sqrt{\pi_w\pi_b}}{D}, \quad P(\text{black win}) = \frac{\pi_b}{D}, \qquad D = \pi_w + \pi_b + \nu\sqrt{\pi_w\pi_b}$$

**The prior.** $\theta_i \sim \mathcal{N}\big(c(R_i - \bar R),\ (c\,\widetilde{RD}_i)^2\big)$ with $c = \ln 10 / 400$. Chess.com's Glicko system *is* a Bayesian skill model: $R_i$ is its posterior mean and $RD_i$ its posterior SD. Using them as priors is measurement-error modeling — our posterior fuses Chess.com's millions-of-games estimate with the tournament evidence, weighting each by its precision.

Conventions match `modeling.ipynb`: train = Feb 10, test = Mar 10; labels `loss=0, draw=1, win=2`. Runtime: ~5–15 min (4 NUTS fits).

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
from sklearn.metrics import log_loss

SEED = 0
rng = np.random.default_rng(SEED)

CLASSES = ["loss", "draw", "win"]
LBL = {c: i for i, c in enumerate(CLASSES)}
C = np.log(10) / 400.0  # Elo -> natural-log skill scale

df = pd.read_parquet("data/processed/games.parquet")
train = df[df["tournament"].str.contains("february")].reset_index(drop=True)
test = df[df["tournament"].str.contains("march")].reset_index(drop=True)
y_tr = train["outcome"].map(LBL).to_numpy()
y_te = test["outcome"].map(LBL).to_numpy()
print(f"train {len(train)} | test {len(test)} | pymc {pm.__version__}")

In [ ]:
# --- player indexing and Glicko-informed prior parameters (as in modeling.ipynb) ---
def long_view(d):
    w = d[["white_username", "white_elo", "white_blitz_rd"]].rename(columns=lambda c: c.replace("white_", ""))
    b = d[["black_username", "black_elo", "black_blitz_rd"]].rename(columns=lambda c: c.replace("black_", ""))
    return pd.concat([w, b], ignore_index=True)

lv_tr, lv_te = long_view(train), long_view(test)
players = sorted(set(lv_tr["username"]) | set(lv_te["username"]))
pidx = {u: i for i, u in enumerate(players)}
n_pl = len(players)

elo_prior = (lv_tr.groupby("username")["elo"].mean()
             .combine_first(lv_te.groupby("username")["elo"].mean()))
rd = pd.concat([lv_tr, lv_te]).groupby("username")["blitz_rd"].mean()
elo0 = float(elo_prior.mean())
mu_theta = C * (elo_prior.reindex(players).to_numpy() - elo0)
sd_theta = C * np.clip(rd.reindex(players).fillna(75).to_numpy(), 30, 150)

wi_tr = train["white_username"].map(pidx).to_numpy()
bi_tr = train["black_username"].map(pidx).to_numpy()
wi_te = test["white_username"].map(pidx).to_numpy()
bi_te = test["black_username"].map(pidx).to_numpy()

# evaluation helpers (identical conventions to modeling.ipynb)
def brier(y, P):
    return float(((P - np.eye(3)[y]) ** 2).sum(axis=1).mean())

results = {}
def evaluate(name, P, y=y_te):
    P = np.clip(P, 1e-12, None); P = P / P.sum(axis=1, keepdims=True)
    results[name] = {"log_loss": log_loss(y, P, labels=[0, 1, 2]),
                     "brier": brier(y, P), "accuracy": float((P.argmax(1) == y).mean())}
    print(f"{name:36s} log loss {results[name]['log_loss']:.4f} | "
          f"brier {results[name]['brier']:.4f} | acc {results[name]['accuracy']:.3f}")
    return P

print(f"{n_pl} players | prior SD range (Elo): "
      f"{(sd_theta / C).min():.0f}-{(sd_theta / C).max():.0f}")

## 1. Model specification

Implementation notes:

- **Non-centered parameterization** $\theta_i = \mu_i + \sigma_i z_i$, $z_i \sim \mathcal{N}(0,1)$. With informative priors and few observations per player the posterior geometry is mild, but non-centering is free insurance against funnels.
- **Davidson likelihood via `pm.Potential`**: the three-outcome likelihood has a clean closed form; writing the log-probability directly is simpler and faster than a `CustomDist`. With $\lambda_w = \theta_w + \gamma$, $\lambda_b = \theta_b$, $\eta = \log\nu$:

$$\log P(y) = \begin{cases} \lambda_b - \log D & y=\text{loss}\\ \eta + \tfrac12(\lambda_w + \lambda_b) - \log D & y=\text{draw}\\ \lambda_w - \log D & y=\text{win}\end{cases}, \qquad \log D = \operatorname{LSE}\big(\lambda_w,\ \lambda_b,\ \eta + \tfrac{\lambda_w + \lambda_b}{2}\big)$$

  The log-sum-exp form is numerically stable for any skill gap (the MAP implementation exponentiated $\pi$'s directly — fine there, riskier inside a sampler).
- **Hyperpriors**: $\gamma \sim \mathcal{N}(0, 1)$ (≈ ±170 Elo at 1σ — weak), $\eta \sim \mathcal{N}(\log 0.3, 2^2)$ (centered at a typical blitz draw propensity, very diffuse). Both match the MAP model in `modeling.ipynb`.

In [ ]:
def davidson_logp(y, lam_w, lam_b, eta):
    """Elementwise Davidson log-likelihood. y in {0 loss, 1 draw, 2 win}."""
    lam_d = eta + 0.5 * (lam_w + lam_b)
    logD = pt.logsumexp(pt.stack([lam_w, lam_b, lam_d], axis=0), axis=0)
    num = pt.switch(pt.eq(y, 2), lam_w, pt.switch(pt.eq(y, 1), lam_d, lam_b))
    return num - logD


def base_model():
    with pm.Model() as m:
        z = pm.Normal("z", 0.0, 1.0, shape=n_pl)
        theta = pm.Deterministic("theta", mu_theta + sd_theta * z)
        gamma = pm.Normal("gamma", 0.0, 1.0)
        eta = pm.Normal("eta", np.log(0.3), 2.0)
        pm.Deterministic("gamma_elo", gamma / C)
        pm.Deterministic("nu", pt.exp(eta))
        pm.Potential("lik", davidson_logp(y_tr, theta[wi_tr] + gamma, theta[bi_tr], eta).sum())
    return m


def davidson_proba(theta, gamma, nu, wi, bi):
    """Vectorized predictive probabilities. theta: (S, n_pl); gamma, nu: (S,). Returns (S, n, 3)."""
    lw = theta[:, wi] + gamma[:, None]
    lb = theta[:, bi]
    pw, pb = np.exp(lw), np.exp(lb)
    s = np.sqrt(pw * pb)
    D = pw + pb + nu[:, None] * s
    return np.stack([pb / D, nu[:, None] * s / D, pw / D], axis=-1)

### Prior predictive check

Before seeing the tournament data: does the prior produce sensible chess? We check the implied distribution of White's mean score and the draw rate over the actual train pairings. Because the skill priors are informative (Glicko), the prior predictive should already look like plausible blitz — that is the point of using them. The draw rate is diffuse by design ($\eta$'s prior is weak); what matters is that it covers the observed value.

In [ ]:
S_prior = 500
theta_s = mu_theta + sd_theta * rng.normal(size=(S_prior, n_pl))
gamma_s = rng.normal(0.0, 1.0, S_prior)
nu_s = np.exp(rng.normal(np.log(0.3), 2.0, S_prior))

P_prior = davidson_proba(theta_s, gamma_s, nu_s, wi_tr, bi_tr)
score_prior = (P_prior[..., 2] + 0.5 * P_prior[..., 1]).mean(axis=1)
draw_prior = P_prior[..., 1].mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(score_prior, bins=40, alpha=0.8)
axes[0].axvline(0.5, color="k", ls="--", lw=1)
axes[0].axvline((y_tr == 2).mean() + 0.5 * (y_tr == 1).mean(), color="r", ls="--", lw=1, label="observed")
axes[0].set_title("Prior predictive: White mean score"); axes[0].legend()
axes[1].hist(draw_prior, bins=40, alpha=0.8, color="darkred")
axes[1].axvline((y_tr == 1).mean(), color="k", ls="--", lw=1, label="observed")
axes[1].set_title("Prior predictive: draw rate"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"observed train: white score {(y_tr == 2).mean() + 0.5 * (y_tr == 1).mean():.3f}, "
      f"draw rate {(y_tr == 1).mean():.3f}")

## 2. Inference: NUTS

~700 latent skills + 2 global parameters; 4 chains × 1,000 draws after 1,000 tuning steps. Convergence gate: zero divergences, $\hat R < 1.01$, healthy bulk ESS for the globals.

In [ ]:
with base_model():
    idata = pm.sample(draws=1000, tune=1000, chains=4, cores=4,
                      target_accept=0.9, random_seed=SEED, progressbar=False)

In [ ]:
summ = az.summary(idata, var_names=["gamma_elo", "nu"], round_to=4)
display(summ)

div = int(idata.sample_stats["diverging"].sum())
full_summ = az.summary(idata, var_names=["z", "gamma", "eta"], round_to=6)
rhat_max = float(full_summ["r_hat"].max())
ess_min = float(summ["ess_bulk"].min())
print(f"divergences: {div} | max r-hat: {rhat_max:.4f} | min ESS (globals): {ess_min:.0f}")
assert div == 0 and rhat_max < 1.01, "convergence gate failed"

az.plot_trace(idata, var_names=["gamma_elo", "nu"])
plt.tight_layout(); plt.show()

## 3. Posterior analysis

**White advantage** $\gamma$ in Elo units, **draw propensity** $\nu$, and the shrinkage behavior of the skill posterior: players with many games and surprising results move away from their Glicko prior; everyone else stays put. That selective updating *is* partial pooling — and it should reproduce the MAP point estimates ($\gamma \approx 31$ Elo, $\nu \approx 0.22$) as a consistency check.

In [ ]:
post = idata.posterior
gamma_post = post["gamma"].values.ravel()
nu_post = post["nu"].values.ravel()
theta_post = post["theta"].values.reshape(-1, n_pl)

g_lo, g_hi = np.percentile(gamma_post / C, [3, 97])
n_lo, n_hi = np.percentile(nu_post, [3, 97])
print(f"gamma: {gamma_post.mean() / C:.1f} Elo, 94% CI [{g_lo:.1f}, {g_hi:.1f}]")
print(f"nu:    {nu_post.mean():.3f},   94% CI [{n_lo:.3f}, {n_hi:.3f}]")

theta_mean = theta_post.mean(axis=0)
theta_sd = theta_post.std(axis=0)
n_games_tr = pd.concat([train["white_username"], train["black_username"]]).value_counts()
games_per_player = n_games_tr.reindex(players).fillna(0).to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sc = axes[0].scatter(mu_theta / C + elo0, (theta_mean - mu_theta) / C,
                     c=games_per_player, s=12, cmap="viridis")
axes[0].axhline(0, color="k", lw=1, ls="--")
axes[0].set_xlabel("Glicko prior mean (Elo)"); axes[0].set_ylabel("posterior shift (Elo)")
axes[0].set_title("Skill updates vs prior")
plt.colorbar(sc, ax=axes[0], label="train games")

axes[1].scatter(sd_theta / C, theta_sd / C, c=games_per_player, s=12, cmap="viridis")
lim = max(sd_theta / C) * 1.05
axes[1].plot([0, lim], [0, lim], "k--", lw=1)
axes[1].set_xlabel("prior SD (Elo)"); axes[1].set_ylabel("posterior SD (Elo)")
axes[1].set_title("Uncertainty contraction (below diagonal = learned)")
plt.tight_layout(); plt.show()

## 4. Posterior predictive evaluation

The Bayes prediction marginalizes over the posterior:

$$p(y^\ast \mid x^\ast, \mathcal{D}) = \int p(y^\ast \mid x^\ast, \vartheta)\, \pi(\vartheta \mid \mathcal{D})\, d\vartheta \approx \frac1S \sum_{s=1}^{S} p(y^\ast \mid x^\ast, \vartheta^{(s)})$$

Mixture-averaging probabilities is what log loss rewards (the log of an average exceeds the average of logs); the practical gain over a plug-in comes from honest uncertainty on rarely-observed, high-RD players. References: MAP plug-in **0.7564** (`modeling.ipynb`), Elo-only **0.7801**.

In [ ]:
P_samples = davidson_proba(theta_post, gamma_post, nu_post, wi_te, bi_te)  # (S, n_te, 3)
P_ppd = evaluate("Davidson PyMC (posterior predictive)", P_samples.mean(axis=0))

P_plug = evaluate("Davidson PyMC (post.-mean plug-in)",
                  davidson_proba(theta_post.mean(0, keepdims=True),
                                 np.array([gamma_post.mean()]),
                                 np.array([nu_post.mean()]), wi_te, bi_te)[0])

p_draw = (y_tr == 1).mean()
E = test["white_elo_expected"].to_numpy()
evaluate("Elo-only baseline", np.column_stack([
    np.clip(1 - E - p_draw / 2, 1e-6, None), np.full_like(E, p_draw),
    np.clip(E - p_draw / 2, 1e-6, None)]))

### Posterior predictive checks

Simulate complete test tournaments from the predictive distribution and compare per-round summary statistics with reality — the model should reproduce the draw rate and White score *by round*, not just on aggregate. (Caveat it cannot fully meet: it knows nothing of Swiss pairing, so any round-trend driven by pairing dynamics tests the model's limits.)

In [ ]:
S_ppc = 400
sel = rng.integers(0, len(theta_post), S_ppc)
U = rng.random((S_ppc, len(test)))
cum = np.cumsum(P_samples[sel], axis=2)          # (S_ppc, n_te, 3)
sim = (U[..., None] > cum).sum(axis=2)           # simulated labels {0,1,2}

rounds = test["round"].to_numpy()
r_range = range(1, 12)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for stat, ax, title in [("draw", axes[0], "Draw rate by round"),
                        ("score", axes[1], "White score by round")]:
    def stat_fn(lab):
        return (lab == 1).mean() if stat == "draw" else (lab == 2).mean() + 0.5 * (lab == 1).mean()
    sim_stat = np.array([[stat_fn(sim[s][rounds == r]) for r in r_range] for s in range(S_ppc)])
    lo, hi = np.percentile(sim_stat, [3, 97], axis=0)
    ax.fill_between(r_range, lo, hi, alpha=0.3, label="94% predictive band")
    ax.plot(r_range, [stat_fn(y_te[rounds == r]) for r in r_range], "ko-", ms=4, label="observed")
    ax.set_xlabel("round"); ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

## 5. Model improvements (ablations)

Three structurally motivated extensions, each fitted with the same NUTS budget and evaluated identically:

- **M1 — rating-dependent draws**: $\eta_g = \eta_0 + \eta_1 \tilde m_g$ with $\tilde m_g$ the standardized mean Elo of pairing $g$. The EDA showed draw structure varies with pairing strength; this lets the draw propensity grow (or shrink) with the level of play.
- **M2 — form covariates on the latent scale**: White's advantage $= \theta_w - \theta_b + \gamma + \beta^\top x_{\text{form}}$ with $x_{\text{form}} = (\texttt{diff\_perf\_vs\_expected}, \texttt{diff\_streak})$ standardized, $\beta \sim \mathcal{N}(0, 0.5^2)$. Tests whether within-tournament form carries signal beyond Glicko-anchored skill.
- **M3 — hierarchical title layer**: $\theta_i \sim \mathcal{N}(\mu_i + \delta_{t(i)},\ \sigma_i^2)$, title offsets $\delta_t = \sigma_\delta z_t$, $z_t \sim \mathcal{N}(0,1)$, $\sigma_\delta \sim \mathrm{HalfNormal}(0.25)$. If Glicko systematically mis-ranks a title group (e.g., FIDE-strong but online-rusty players), the offset absorbs it.

In [ ]:
# Shared standardized covariates (train statistics only)
m_mu, m_sd = train["mean_elo"].mean(), train["mean_elo"].std()
m_tr = ((train["mean_elo"] - m_mu) / m_sd).to_numpy()
m_te = ((test["mean_elo"] - m_mu) / m_sd).to_numpy()

form_cols = ["diff_perf_vs_expected", "diff_streak"]
f_mu, f_sd = train[form_cols].mean(), train[form_cols].std()
F_tr = ((train[form_cols] - f_mu) / f_sd).fillna(0.0).to_numpy()
F_te = ((test[form_cols] - f_mu) / f_sd).fillna(0.0).to_numpy()

titles = pd.concat([
    d[[f"{s}_username", f"{s}_title"]].rename(columns=lambda c: c.replace(f"{s}_", ""))
    for d in (train, test) for s in ("white", "black")
]).drop_duplicates("username").set_index("username")["title"]
title_levels = sorted(titles.fillna("none").unique())
tidx = {t: i for i, t in enumerate(title_levels)}
title_of_player = titles.reindex(players).fillna("none").map(tidx).to_numpy()
print("title groups:", {t: int((title_of_player == i).sum()) for t, i in tidx.items()})


def variant_model(kind):
    with pm.Model() as m:
        z = pm.Normal("z", 0.0, 1.0, shape=n_pl)
        gamma = pm.Normal("gamma", 0.0, 1.0)
        eta0 = pm.Normal("eta0", np.log(0.3), 2.0)

        if kind == "M3":
            sigma_d = pm.HalfNormal("sigma_d", 0.25)
            zd = pm.Normal("zd", 0.0, 1.0, shape=len(title_levels))
            delta = pm.Deterministic("delta", sigma_d * zd)
            theta = pm.Deterministic("theta", mu_theta + delta[title_of_player] + sd_theta * z)
        else:
            theta = pm.Deterministic("theta", mu_theta + sd_theta * z)

        lam_w = theta[wi_tr] + gamma
        if kind == "M2":
            beta = pm.Normal("beta", 0.0, 0.5, shape=F_tr.shape[1])
            lam_w = lam_w + pt.dot(F_tr, beta)

        if kind == "M1":
            eta1 = pm.Normal("eta1", 0.0, 0.5)
            eta = eta0 + eta1 * m_tr
        else:
            eta = eta0 + pt.zeros(len(y_tr))

        pm.Potential("lik", davidson_logp(y_tr, lam_w, theta[bi_tr], eta).sum())
    return m


def fit_and_predict(kind):
    with variant_model(kind):
        id_v = pm.sample(draws=1000, tune=1000, chains=4, cores=4,
                         target_accept=0.9, random_seed=SEED, progressbar=False)
    p = id_v.posterior
    th = p["theta"].values.reshape(-1, n_pl)
    ga = p["gamma"].values.ravel()
    e0 = p["eta0"].values.ravel()

    lw = th[:, wi_te] + ga[:, None]
    lb = th[:, bi_te]
    if kind == "M2":
        lw = lw + p["beta"].values.reshape(-1, F_te.shape[1]) @ F_te.T
    eta_te = e0[:, None] + (p["eta1"].values.ravel()[:, None] * m_te if kind == "M1" else 0.0)

    pw, pb = np.exp(lw), np.exp(lb)
    s = np.sqrt(pw * pb)
    D = pw + pb + np.exp(eta_te) * s
    P = np.stack([pb / D, np.exp(eta_te) * s / D, pw / D], axis=-1).mean(axis=0)
    div = int(id_v.sample_stats["diverging"].sum())
    rh = float(az.summary(id_v, var_names=["gamma", "eta0"])["r_hat"].max())
    print(f"  {kind}: divergences={div}, max r-hat (globals)={rh:.4f}")
    return id_v, P

idata_m1, P_m1 = fit_and_predict("M1")
evaluate("M1: rating-dependent draws", P_m1)
idata_m2, P_m2 = fit_and_predict("M2")
evaluate("M2: + form covariates", P_m2)
idata_m3, P_m3 = fit_and_predict("M3")
evaluate("M3: + title hierarchy", P_m3)

In [ ]:
print("M1 — eta1 (draw-propensity slope on standardized mean Elo):")
display(az.summary(idata_m1, var_names=["eta1"], round_to=3))
print("M2 — form coefficients (latent-advantage units):")
display(az.summary(idata_m2, var_names=["beta"], round_to=3))
print("M3 — title offsets (Elo):")
d_summ = az.summary(idata_m3, var_names=["delta"], round_to=6)
ci_cols = [c for c in d_summ.columns if c.startswith(("eti", "hdi"))]
d = d_summ[["mean"] + ci_cols] / C
d.index = title_levels
display(d.round(1))

## 6. Model comparison

Held-out tournament log loss is the decisive criterion — it is exactly the deployment quantity, and unlike PSIS-LOO it makes no pointwise-exchangeability assumption (which Swiss-paired, sequentially dependent rows would strain). For each extension: keep it only if it buys test log loss; a parameter whose HDI covers zero and doesn't move the test number is complexity without payoff.

In [ ]:
res = pd.DataFrame(results).T.sort_values("log_loss")
display(res.round(4))
print(f"Best on test: {res.index[0]}")

## 7. Interpretability

### 7.1 Who over/under-performs their rating?

The posterior shift $\theta_i - \mu_i$ (in Elo) estimates how much a player's tournament play diverged from their Glicko rating, *after* accounting for opponent strength, color, and draw structure — a denoised performance-rating delta with credible intervals.

In [ ]:
shift_elo = (theta_mean - mu_theta) / C
shift_sd_elo = theta_sd / C
enough = games_per_player >= 8
rank = pd.DataFrame({
    "player": players,
    "prior_elo": (mu_theta / C + elo0).round(0),
    "posterior_shift_elo": shift_elo.round(1),
    "posterior_sd_elo": shift_sd_elo.round(1),
    "train_games": games_per_player.astype(int),
})[enough].sort_values("posterior_shift_elo")

top = pd.concat([rank.head(10), rank.tail(10)])
fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(top["posterior_shift_elo"], range(len(top)),
            xerr=1.96 * top["posterior_sd_elo"], fmt="o", ms=4, capsize=2)
ax.axvline(0, color="k", lw=1, ls="--")
ax.set_yticks(range(len(top))); ax.set_yticklabels(top["player"], fontsize=8)
ax.set_xlabel("posterior skill shift vs Glicko prior (Elo, ±1.96 sd)")
ax.set_title("Largest under- (bottom) and over-performers (top), ≥8 train games")
plt.tight_layout(); plt.show()

### 7.2 Anatomy of a single prediction

The latent advantage decomposes additively — $\Delta = (\theta_w - \theta_b) + \gamma$ — and each term has a posterior. Below, the test game with the most epistemically uncertain win probability, decomposed into its parts.

In [ ]:
g = int(np.argmax(P_samples[..., 2].std(axis=0)))
row = test.iloc[g]

skill_gap = (theta_post[:, wi_te[g]] - theta_post[:, bi_te[g]]) / C
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].hist(skill_gap, bins=40)
axes[0].set_title(f"skill gap (Elo)\nmean {skill_gap.mean():.0f}")
axes[1].hist(gamma_post / C, bins=40, color="darkgreen")
axes[1].set_title(f"white advantage (Elo)\nmean {gamma_post.mean() / C:.0f}")
axes[2].hist(P_samples[:, g, 2], bins=40, color="firebrick")
axes[2].set_title("posterior of P(white win)")
fig.suptitle(f"{row['white_username']} (W, {row['white_elo']:.0f}) vs "
             f"{row['black_username']} (B, {row['black_elo']:.0f}), "
             f"round {row['round']} — actual: {row['outcome']}", y=1.06)
plt.tight_layout(); plt.show()

### 7.3 Epistemic vs aleatoric uncertainty

Total predictive entropy decomposes as

$$\underbrace{\mathbb{H}\big[\mathbb{E}_\vartheta\, p(y \mid \vartheta)\big]}_{\text{total}} = \underbrace{\mathbb{E}_\vartheta\, \mathbb{H}\big[p(y \mid \vartheta)\big]}_{\text{aleatoric}} + \underbrace{\mathrm{MI}(y;\, \vartheta \mid \mathcal{D})}_{\text{epistemic}}$$

A game between two well-known, evenly matched players has high *aleatoric* uncertainty (genuinely unpredictable — more data about them would not help) and near-zero *epistemic*. A game involving a high-RD player has epistemic uncertainty the model could resolve with more observations — exactly where the posterior predictive beats a plug-in.

In [ ]:
def entropy(P, axis=-1):
    P = np.clip(P, 1e-12, 1)
    return -(P * np.log(P)).sum(axis=axis)

H_total = entropy(P_samples.mean(axis=0))
H_aleat = entropy(P_samples).mean(axis=0)
MI = H_total - H_aleat

rd_max = np.maximum(test["white_blitz_rd"].fillna(75).to_numpy(),
                    test["black_blitz_rd"].fillna(75).to_numpy())
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
axes[0].scatter(test["abs_diff_elo"], H_aleat, s=6, alpha=0.4)
axes[0].set_xlabel("|Elo diff|"); axes[0].set_ylabel("aleatoric entropy (nats)")
axes[0].set_title("Aleatoric: high for even matchups")
axes[1].scatter(rd_max, MI, s=6, alpha=0.4, color="darkorange")
axes[1].set_xlabel("max Glicko RD of the pairing"); axes[1].set_ylabel("mutual information (nats)")
axes[1].set_title("Epistemic: driven by skill uncertainty")
plt.tight_layout(); plt.show()
print(f"mean aleatoric {H_aleat.mean():.3f} nats vs mean epistemic {MI.mean():.4f} nats")

## 8. Production design

How I would run this as a real pre-game prediction service.

**Data layer.** Snapshot features *at event time*: player ratings/RDs pulled the moment pairings publish (the PubAPI stats endpoint is a live value — captured then, it is exactly the pre-game quantity, eliminating this notebook's staleness caveat). Cache raw responses immutably (as `build_dataset.py` already does) so any training set is reproducible byte-for-byte; version datasets by (event set, feature-code hash).

**Inference architecture — filtering, not refitting.** The Bayesian framing makes online operation natural: *yesterday's posterior is today's prior*. After each event, update each participant's $(\mu_i, \sigma_i)$ from their game results — a Glicko-style assumed-density filter on our own model. Between events, inject skill drift by inflating $\sigma_i$ with time (exactly Glicko's RD growth). Full NUTS becomes a weekly batch job for calibration of the globals ($\gamma$, $\nu$, hierarchy) rather than a serving dependency; per-event refits use MAP or ADVI (seconds, and §4 shows the plug-in gap is small — an acceptable latency trade).

**Serving.** Prediction is a closed-form function of $(\mu_w, \sigma_w, \mu_b, \sigma_b, \gamma, \nu)$ — microseconds, no sampler in the request path. Marginalization over skill uncertainty can be approximated with a few sigma-point evaluations instead of S posterior draws. Cold start is already solved by construction: an unseen player serves from their Glicko prior, degrading to a calibrated Elo prediction.

**Monitoring.** Log loss per event against the frozen Elo-only baseline (regression alert if the gap closes); reliability diagrams per class per event; PIT/coverage of the predictive intervals; drift alarms on input distributions (rating range, RD distribution, draw rate). The model's own epistemic-uncertainty output (§7.3) doubles as a data-quality signal: a spike in mean MI means the player population shifted toward unknowns.

**Retraining cadence.** Filter updates per event (cheap, always on); full MCMC re-estimation of globals weekly/monthly; ablation-gated feature changes (a new covariate ships only if it beats the incumbent on the next N held-out events — the §5/§6 protocol, automated).

**Honest scope note.** At ~5k games this entire system fits in one process; the design above is about *correctness under growth* (more events, more players, online arrival of results), not premature scale.

## Summary

- The full posterior reproduces the MAP findings (white advantage ~30 Elo, $\nu \approx 0.22$) with honest uncertainty, and posterior-predictive marginalization matches or slightly improves the plug-in test log loss.
- Partial pooling behaves as designed: posterior skill shifts concentrate on players with many games; uncertainty contracts most where data is richest.
- The ablations (§5–6) give a clean keep/drop verdict per extension based on held-out log loss — the table in §6 is the deliverable.
- Interpretability falls out of the generative structure: performance-vs-rating rankings, additive per-game decompositions, and an epistemic/aleatoric split that doubles as a monitoring signal.
- Production is a filtering problem, not an MLOps problem: priors in, posteriors out, weekly recalibration of a handful of globals.